In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructField, StructType, StringType, DateType, IntegerType
from pyspark.sql.functions import *

spark = SparkSession.builder.appName("Retail_dat_processing").getOrCreate()

In [ ]:
# 1. Read the source CSV. --> done

# Display:

# First 10 records --> done
# Schema  --> done
# Number of records --> done
sales_schema = StructType([ StructField('order_id', StringType()),
                          StructField('order_date', DateType()),
                          StructField('customer_id', StringType()),
                          StructField('customer_name', StringType()),
                          StructField('product_id', StringType()),
                          StructField('product_name', StringType()),
                          StructField('category', StringType()),
                          StructField('store_id', StringType()),
                          StructField('store_name', StringType()),
                          StructField('city', StringType()),
                          StructField('sate', StringType()),
                          StructField('quantity', IntegerType()),
                          StructField('unit_price', IntegerType()),
                          StructField('discount_pct', IntegerType())])


input_df = spark.read.csv('sales_data.csv', header= True, schema=sales_schema)

# input_df = spark.read.csv('sales_data.csv', header= True, inferSchema=True)
# in prod, we should not use inferSchema=True for csv file ==> does not give good performance

input_df.show(10, False)
input_df.printSchema()
input_df.count()

+----------+----------+-----------+---------------+----------+------------+-----------+--------+-------------+---------+-----------+--------+----------+------------+
|order_id  |order_date|customer_id|customer_name  |product_id|product_name|category   |store_id|store_name   |city     |sate       |quantity|unit_price|discount_pct|
+----------+----------+-----------+---------------+----------+------------+-----------+--------+-------------+---------+-----------+--------+----------+------------+
|ORD0000001|2026-05-31|CUST22129  |James Brown    |P003      |Headphones  |Electronics|S005    |Main Store   |Pune     |Maharashtra|8       |3000      |5           |
|ORD0000002|2026-03-07|CUST29205  |Mia Thomas     |P005      |Office Chair|Furniture  |S005    |Main Store   |Pune     |Maharashtra|9       |7500      |5           |
|ORD0000003|2026-03-29|CUST19584  |Daniel Martin  |P010      |Bookshelf   |Furniture  |S003    |City Store   |Chennai  |Tamil Nadu |10      |6000      |5           |
|ORD

500000

In [ ]:
# 2. Create a new column called gross_amount. --> done
# Formula: quantity × unit_price

gross_df = input_df.withColumn("gross_amount", input_df.quantity * input_df.unit_price)
gross_df.show(10, False)
gross_df.printSchema()

+----------+----------+-----------+---------------+----------+------------+-----------+--------+-------------+---------+-----------+--------+----------+------------+------------+
|order_id  |order_date|customer_id|customer_name  |product_id|product_name|category   |store_id|store_name   |city     |sate       |quantity|unit_price|discount_pct|gross_amount|
+----------+----------+-----------+---------------+----------+------------+-----------+--------+-------------+---------+-----------+--------+----------+------------+------------+
|ORD0000001|2026-05-31|CUST22129  |James Brown    |P003      |Headphones  |Electronics|S005    |Main Store   |Pune     |Maharashtra|8       |3000      |5           |24000       |
|ORD0000002|2026-03-07|CUST29205  |Mia Thomas     |P005      |Office Chair|Furniture  |S005    |Main Store   |Pune     |Maharashtra|9       |7500      |5           |67500       |
|ORD0000003|2026-03-29|CUST19584  |Daniel Martin  |P010      |Bookshelf   |Furniture  |S003    |City Stor

In [ ]:
# 3. Create discount_amount.
# Formula: gross_amount * discount_pct / 100

discount_df = gross_df.withColumn('discount_amount',  gross_df.gross_amount * gross_df.discount_pct / 100)
discount_df.show(10, False)

+----------+----------+-----------+---------------+----------+------------+-----------+--------+-------------+---------+-----------+--------+----------+------------+------------+---------------+
|order_id  |order_date|customer_id|customer_name  |product_id|product_name|category   |store_id|store_name   |city     |sate       |quantity|unit_price|discount_pct|gross_amount|discount_amount|
+----------+----------+-----------+---------------+----------+------------+-----------+--------+-------------+---------+-----------+--------+----------+------------+------------+---------------+
|ORD0000001|2026-05-31|CUST22129  |James Brown    |P003      |Headphones  |Electronics|S005    |Main Store   |Pune     |Maharashtra|8       |3000      |5           |24000       |1200.0         |
|ORD0000002|2026-03-07|CUST29205  |Mia Thomas     |P005      |Office Chair|Furniture  |S005    |Main Store   |Pune     |Maharashtra|9       |7500      |5           |67500       |3375.0         |
|ORD0000003|2026-03-29|CU

In [ ]:
# 4. Create a new column final_amount.
# Formula: gross_amount - discount_amount

final_df = discount_df.withColumn('final_amount', discount_df.gross_amount - discount_df.discount_amount)
final_df.show(10, False)

+----------+----------+-----------+---------------+----------+------------+-----------+--------+-------------+---------+-----------+--------+----------+------------+------------+---------------+------------+
|order_id  |order_date|customer_id|customer_name  |product_id|product_name|category   |store_id|store_name   |city     |sate       |quantity|unit_price|discount_pct|gross_amount|discount_amount|final_amount|
+----------+----------+-----------+---------------+----------+------------+-----------+--------+-------------+---------+-----------+--------+----------+------------+------------+---------------+------------+
|ORD0000001|2026-05-31|CUST22129  |James Brown    |P003      |Headphones  |Electronics|S005    |Main Store   |Pune     |Maharashtra|8       |3000      |5           |24000       |1200.0         |22800.0     |
|ORD0000002|2026-03-07|CUST29205  |Mia Thomas     |P005      |Office Chair|Furniture  |S005    |Main Store   |Pune     |Maharashtra|9       |7500      |5           |675

In [ ]:
# 5. Rename columns.

# customer_id     → customer_key
# product_id      → product_key
# store_id        → store_key
# unit_price      → selling_price
# discount_pct    → discount_percentage

# renamed_df = final_df.withColumnRenamed('customer_id', 'customer_key') \
#                       .withColumnRenamed('product_id', 'product_key')  \
#                       .withColumnRenamed('store_id', 'store_key') \
#                       .withColumnRenamed('unit_price', 'selling_price') \
#                       .withColumnRenamed('discount_pct', 'discount_percentage')

renamed_df = final_df.withColumnsRenamed({'customer_id' : 'customer_key',
                                          'product_id' : 'product_key',
                                          'store_id': 'store_key',
                                          'unit_price': 'selling_price',
                                          'discount_pct': 'discount_percentage'})

renamed_df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- customer_key: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product_key: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- store_key: string (nullable = true)
 |-- store_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- sate: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- selling_price: integer (nullable = true)
 |-- discount_percentage: integer (nullable = true)
 |-- gross_amount: integer (nullable = true)
 |-- discount_amount: double (nullable = true)
 |-- final_amount: double (nullable = true)



In [ ]:
# 6. Create store-level sales summary.

# For every store calculate:

# total_orders
# total_quantity
# total_gross_sales
# total_discount
# total_sales
# average_sales
# minimum_sales
# maximum_sales

store_level_summary_df = renamed_df.groupBy("store_key").agg(
    count('order_id').alias("total_orders"),
    sum('quantity').alias("total_quantity"),
    sum('gross_amount').alias("totalgross_sales"),
    sum('discount_amount').alias("total_discount"),
    sum('final_amount').alias("total_sales"),
    avg('final_amount').alias("average_sales"),
    min('final_amount').alias("minimum_sales"),
    max('final_amount').alias("maximum_sales"),
)

store_level_summary_df.show()

+---------+------------+--------------+----------------+--------------+-------------+-----------------+-------------+-------------+
|store_key|total_orders|total_quantity|totalgross_sales|total_discount|  total_sales|    average_sales|minimum_sales|maximum_sales|
+---------+------------+--------------+----------------+--------------+-------------+-----------------+-------------+-------------+
|     S004|       62319|        342712|      4903665600|  3.53716965E8|4.549948635E9|73010.61690656141|        640.0|     550000.0|
|     S001|       62865|        347260|      4961869800|   3.5782037E8| 4.60404943E9|73237.08629603118|        640.0|     550000.0|
|     S008|       62650|        344584|      4982859800|  3.59349485E8|4.623510315E9| 73799.0473264166|        640.0|     550000.0|
|     S002|       62534|        344707|      4997409500|  3.60713785E8|4.636695715E9|74146.79558320274|        640.0|     550000.0|
|     S005|       62468|        343455|      4929393600|   3.5669485E8| 4.57

In [ ]:
# 7. Create category-level summary.

# For every category calculate:

# total_quantity
# total_sales
# average_sales

category_level_summary_df = renamed_df.groupBy("category").agg(
    sum('quantity').alias("total_quantity"),
    sum('final_amount').alias("total_sales"),
    avg('final_amount').alias("average_sales")
)

category_level_summary_df.show()

+-----------+--------------+-------------+------------------+
|   category|total_quantity|  total_sales|     average_sales|
+-----------+--------------+-------------+------------------+
|Electronics|       1375312|2.96492138E10|118617.73192081807|
|  Furniture|        827651|6.520833225E9| 43401.33265666079|
|Accessories|        550146|   5.875369E8|5887.2022765759175|
+-----------+--------------+-------------+------------------+



In [ ]:
# 8. Create product-level summary.

# For every product calculate:
# total_quantity
# total_sales

# Sort products by highest sales.


product_level_summary_df = renamed_df.groupBy("product_name").agg(
    sum('quantity').alias("total_quantity"),
    sum('final_amount').alias("total_sales")
)


# withColumn()  --> if the column does not present in DF, it will create it.
      # otherwise the data will overwrite in existing column
# processed_sales_df = product_level_summary_df.withColumn("total_sales", format_number('total_sales', 2)).orderBy(desc("total_sales"))

processed_sales_df = product_level_summary_df.orderBy(desc("total_sales"))
processed_sales_df.show()

+------------+--------------+--------------+
|product_name|total_quantity|   total_sales|
+------------+--------------+--------------+
|      Laptop|        276617|1.411805175E10|
|Mobile Phone|        273946|   6.3521075E9|
|     Monitor|        273387|   4.5651987E9|
|     Printer|        276665|  3.84939825E9|
|Office Table|        275631|   3.0661866E9|
|Office Chair|        274072| 1.907108625E9|
|   Bookshelf|        277948|    1.547538E9|
|  Headphones|        274697|    7.644576E8|
|    Keyboard|        276175|    3.842613E8|
|       Mouse|        273971|    2.032756E8|
+------------+--------------+--------------+



In [ ]:
# 9. Write the processed transaction data.
# Output: processed_sales
# Format: CSV

processed_sales_df.write.mode('ignore').csv('processed_sales', header=True)